# Final 9 visualizations — interactive implementation

Interactive Altair charts for the **9 designs** selected in `baby_names_9_viz_group_G_.ipynb` (3 per problem).

| Problem | Designs |
|---|---|
| **1 — Temporal** | 1.6 Leaf timeline · 1.8 Name genealogy · 1.2 Bump chart |
| **2 — Regional** | 2.1 Choropleth · 2.9 Regional flower · 2.4 Department ranking |
| **3 — Gender** | 3.3 Baby ratio · 3.8 Gender trajectory · 3.6 Popularity × mixedness |


In [1]:
import json
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output

alt.data_transformers.disable_max_rows()

# ── Load & clean data ──────────────────────────────────────────────────────
df = pd.read_csv('../data/dpt2020.csv', sep=';', dtype={'annais': str, 'dpt': str, 'nombre': str})
df = df[(df['annais'] != 'XXXX') & (df['preusuel'] != '_PRENOMS_RARES')]
df['annais'] = pd.to_numeric(df['annais'], errors='coerce')
df['nombre'] = pd.to_numeric(df['nombre'], errors='coerce')
df = df.dropna(subset=['annais', 'nombre'])
df['annais'] = df['annais'].astype(int)
df['decade'] = (df['annais'] // 10) * 10

nat = df.groupby(['preusuel', 'annais', 'sexe'])['nombre'].sum().reset_index()
nat['decade'] = (nat['annais'] // 10) * 10
nat_dec = nat.groupby(['preusuel', 'decade', 'sexe'])['nombre'].sum().reset_index()
nat_dec['rank'] = nat_dec.groupby(['decade', 'sexe'])['nombre'].rank(ascending=False, method='min').astype(int)
nat_dec['sex_label'] = nat_dec['sexe'].map({1: 'Male', 2: 'Female'})

pop_total = nat.groupby(['preusuel', 'annais'])['nombre'].sum().reset_index()
top50 = df.groupby('preusuel')['nombre'].sum().nlargest(50).index.tolist()
top20_map = top50[:20]

# Map / regional aggregates
total_dpt = df.groupby(['dpt', 'annais'])['nombre'].sum().reset_index()
total_dpt['decade'] = (total_dpt['annais'] // 10) * 10
total_dpt_dec = total_dpt.groupby(['dpt', 'decade'])['nombre'].sum().reset_index()
total_dpt_dec.columns = ['dpt', 'decade', 'total_births']

name_dpt = df.groupby(['preusuel', 'dpt', 'annais'])['nombre'].sum().reset_index()
name_dpt['decade'] = (name_dpt['annais'] // 10) * 10
name_dpt_dec = name_dpt.groupby(['preusuel', 'dpt', 'decade'])['nombre'].sum().reset_index()
name_dpt_dec = name_dpt_dec.merge(total_dpt_dec, on=['dpt', 'decade'], how='left')
name_dpt_dec['rate_per_10k'] = (name_dpt_dec['nombre'] / name_dpt_dec['total_births'] * 10000).round(2)
map_data = name_dpt_dec[name_dpt_dec['preusuel'].isin(top50)].copy()
decades_map = sorted(map_data['decade'].unique().tolist())

with open('../data/departements.geojson') as fgeo:
    geo = json.load(fgeo)
geo_inline = alt.InlineData(values=geo['features'], format=alt.DataFormat(type='json'))
dpt_names_full = {f['properties']['code']: f['properties']['nom'] for f in geo['features']}
dpt_names_full.update({
    '971': 'Guadeloupe', '972': 'Martinique', '973': 'Guyane',
    '974': 'La Réunion', '976': 'Mayotte', '20': 'Corse (ancien)',
})

region_map = {
    'Île-de-France': ['75', '77', '78', '91', '92', '93', '94', '95'],
    'Bretagne': ['22', '29', '35', '56'],
    'PACA': ['04', '05', '06', '13', '83', '84'],
    'Auvergne-RA': ['01', '03', '07', '15', '26', '38', '42', '43', '63', '69', '73', '74'],
    'Occitanie': ['09', '11', '12', '30', '31', '32', '34', '46', '48', '65', '66', '81', '82'],
    'Nouvelle-Aquit.': ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87'],
    'Grand-Est': ['08', '10', '51', '52', '54', '55', '57', '67', '68', '88'],
    'Normandie': ['14', '27', '50', '61', '76'],
    'Hauts-de-France': ['02', '59', '60', '62', '80'],
    'Pays-de-la-Loire': ['44', '49', '53', '72', '85'],
}
dpt_to_reg = {dpt: reg for reg, dpts in region_map.items() for dpt in dpts}
map_data['region'] = map_data['dpt'].map(dpt_to_reg)
reg_agg = (
    map_data.dropna(subset=['region'])
    .groupby(['preusuel', 'region', 'decade'])
    .agg(nombre=('nombre', 'sum'), total_births=('total_births', 'sum'))
    .reset_index()
)
reg_agg['rate_per_10k'] = (reg_agg['nombre'] / reg_agg['total_births'] * 10000).round(2)

# Gender aggregates
gender_nat = nat.groupby(['preusuel', 'sexe'])['nombre'].sum().reset_index()
g_pivot = gender_nat.pivot_table(index='preusuel', columns='sexe', values='nombre', fill_value=0).reset_index()
g_pivot.columns.name = None
g_pivot = g_pivot.rename(columns={1: 'male', 2: 'female'})
for col in ('male', 'female'):
    if col not in g_pivot.columns:
        g_pivot[col] = 0
g_pivot['total'] = g_pivot['male'] + g_pivot['female']
g_pivot['female_ratio'] = g_pivot['female'] / g_pivot['total']
g_pivot['mixedness'] = (g_pivot['female_ratio'] - 0.5).abs()

gender_dec = nat.groupby(['preusuel', 'sexe', 'decade'])['nombre'].sum().reset_index()
g_piv_dec = gender_dec.pivot_table(index=['preusuel', 'decade'], columns='sexe', values='nombre', fill_value=0).reset_index()
g_piv_dec.columns.name = None
g_piv_dec = g_piv_dec.rename(columns={1: 'male', 2: 'female'})
for col in ('male', 'female'):
    if col not in g_piv_dec.columns:
        g_piv_dec[col] = 0
g_piv_dec['total'] = g_piv_dec['male'] + g_piv_dec['female']
g_piv_dec['female_ratio'] = g_piv_dec['female'] / g_piv_dec['total']
g_piv_dec['gender_type'] = 'Mixed (both)'
g_piv_dec.loc[g_piv_dec['female_ratio'] > 0.8, 'gender_type'] = 'Mostly female'
g_piv_dec.loc[g_piv_dec['female_ratio'] < 0.2, 'gender_type'] = 'Mostly male'

unisex_names = g_pivot[(g_pivot['total'] >= 3000) & (g_pivot['mixedness'] <= 0.25)].sort_values('mixedness')['preusuel'].tolist()
mixed_top = g_pivot[g_pivot['total'] >= 5000].nsmallest(20, 'mixedness')['preusuel'].tolist()

print(f'{len(df):,} rows · {df["preusuel"].nunique():,} names · {df["annais"].min()}–{df["annais"].max()}')


3,773,985 rows · 16,226 names · 1900–2022


# Problem 1 — Temporal evolution

## 1.6 — Leaf timeline / visual popularity frieze

In [2]:
# Top-10 names on x-axis; decades on y-axis; one vertical vine per name; leaf marks at each decade
LEAF_SHAPE = 'M0,-7 C3,-4 5,1 0,8 C-5,1 -3,-4 0,-7 Z'

def _leaf_angle(name, decade):
    # Stable pseudo-random tilt per leaf (roughly ±55°)
    h = hash((name, int(decade))) % 110
    return h - 55

def make_leaf_frieze(sex_label):
    sex_code = 2 if sex_label == 'Female' else 1
    top10 = (
        nat[nat['sexe'] == sex_code]
        .groupby('preusuel')['nombre'].sum()
        .nlargest(10).index.tolist()
    )
    d = nat_dec[(nat_dec['sexe'] == sex_code) & (nat_dec['preusuel'].isin(top10))].copy()
    d['norm'] = d.groupby('preusuel')['nombre'].transform(lambda x: x / x.max())
    d['leaf_angle'] = [_leaf_angle(n, dec) for n, dec in zip(d['preusuel'], d['decade'])]

    vines = alt.Chart(d).mark_line(color='#689f38', strokeWidth=2.5, opacity=0.75).encode(
        x=alt.X('preusuel:N', sort=top10, title='Name (top 10)'),
        y=alt.Y('decade:O', title='Decade', sort='descending'),
        order=alt.Order('decade:Q'),
        detail='preusuel:N',
    )
    leaves = alt.Chart(d).mark_point(
        filled=True, shape=LEAF_SHAPE, stroke='#1b5e20', strokeWidth=0.35,
    ).encode(
        x=alt.X('preusuel:N', sort=top10),
        y=alt.Y('decade:O', sort='descending'),
        angle=alt.Angle('leaf_angle:Q', scale=None),
        size=alt.Size('nombre:Q', scale=alt.Scale(range=[40, 520]), title='Births'),
        color=alt.Color('norm:Q', scale=alt.Scale(scheme='greens'), title='Relative popularity'),
        tooltip=[
            'preusuel:N', alt.Tooltip('decade:O', title='Decade'),
            alt.Tooltip('nombre:Q', title='Births', format=','),
            alt.Tooltip('norm:Q', title='Relative', format='.2f'),
        ],
    )
    return (vines + leaves).properties(
        width=820, height=560,
        title=alt.Title(
            f'1.6 — Leaf timeline / popularity frieze ({sex_label})',
            subtitle='Each column = one name vine along decades. Leaf size = births; leaf color = relative popularity.',
        ),
    )

leaf_sex = widgets.RadioButtons(
    options=['Female', 'Male'], value='Female', description='Sex:',
)
leaf_out = widgets.Output()

def on_leaf_change(change):
    with leaf_out:
        clear_output(wait=True)
        display(make_leaf_frieze(leaf_sex.value))

leaf_sex.observe(on_leaf_change, names='value')
display(leaf_sex, leaf_out)
on_leaf_change(None)


RadioButtons(description='Sex:', options=('Female', 'Male'), value='Female')

Output()

## 1.8 — Name genealogy / tree

In [3]:
# "Dynasty" names: in top 8 for ≥ 5 decades — connected across generations
dynasty = (
    nat_dec[nat_dec['rank'] <= 8]
    .groupby(['preusuel', 'sex_label'])['decade']
    .nunique()
    .reset_index(name='n_decades')
)
dynasty = dynasty[dynasty['n_decades'] >= 5]['preusuel'].tolist()
gene_data = nat_dec[(nat_dec['preusuel'].isin(dynasty)) & (nat_dec['rank'] <= 8)].copy()

gene_highlight = alt.selection_point(fields=['preusuel'], on='click', toggle=True, empty=True)
gene_sex = alt.param(name='gene_sex', value='Female',
    bind=alt.binding_radio(options=['Female', 'Male'], name='Sex: '))

gene_base = alt.Chart(gene_data).transform_filter('datum.sex_label === gene_sex').encode(
    x=alt.X('decade:O', title='Decade (generation)'),
    y=alt.Y('rank:Q', title='Rank within decade', scale=alt.Scale(reverse=True, domain=[1, 8])),
    color=alt.Color('preusuel:N', legend=alt.Legend(title='Name dynasty (click to highlight)')),
    opacity=alt.condition(gene_highlight, alt.value(1.0), alt.value(0.15)),
    size=alt.condition(gene_highlight, alt.value(4), alt.value(1.5)),
    detail='preusuel:N',
    tooltip=['preusuel:N', 'decade:O', 'rank:Q', alt.Tooltip('nombre:Q', format=',')],
)

last_gene_dec = int(gene_data['decade'].max())
gene_labels = alt.Chart(gene_data[gene_data['decade'] == last_gene_dec]).transform_filter(
    'datum.sex_label === gene_sex'
).mark_text(align='left', dx=6, fontSize=10, fontWeight='bold').encode(
    x='decade:O', y='rank:Q', text='preusuel:N', color='preusuel:N',
    opacity=alt.condition(gene_highlight, alt.value(1.0), alt.value(0.15)),
)

(gene_base.mark_line(point=alt.OverlayMarkDef(size=70)) + gene_labels
).add_params(gene_highlight, gene_sex).properties(
    width=740, height=400,
    title=alt.Title('1.8 — Name genealogy / tree',
                    subtitle='Long-running top-8 names linked across decades. Click a branch to highlight.'),
)


alt.LayerChart(...)

## 1.2 — Bump chart of ranks

In [4]:
bump_data = nat_dec[nat_dec['rank'] <= 20].copy()
bump10 = bump_data[bump_data['rank'] <= 10].copy()

bump_highlight = alt.selection_point(fields=['preusuel'], on='click', toggle=True, empty=True)
bump_sex = alt.param(name='bump_sex', value='Female',
    bind=alt.binding_radio(options=['Female', 'Male'], name='Sex: '))
last_bump_dec = int(bump10['decade'].max())
color_b = alt.Color('preusuel:N', legend=alt.Legend(title='Name (click to highlight)'))

bump_base = alt.Chart(bump10).transform_filter('datum.sex_label === bump_sex').encode(
    x=alt.X('decade:O', title='Decade'),
    y=alt.Y('rank:Q', title='Rank', scale=alt.Scale(reverse=True, domain=[1, 10])),
    color=color_b,
    opacity=alt.condition(bump_highlight, alt.value(1.0), alt.value(0.12)),
    size=alt.condition(bump_highlight, alt.value(4), alt.value(2)),
    detail='preusuel:N',
    tooltip=['preusuel:N', 'decade:O', 'rank:Q', alt.Tooltip('nombre:Q', format=',')],
)
bump_labels = alt.Chart(bump10[bump10['decade'] == last_bump_dec]).transform_filter(
    'datum.sex_label === bump_sex'
).mark_text(align='left', dx=7, fontSize=11, fontWeight='bold').encode(
    x='decade:O', y='rank:Q', text='preusuel:N', color=color_b,
    opacity=alt.condition(bump_highlight, alt.value(1.0), alt.value(0.12)),
)

(bump_base.mark_line(point=alt.OverlayMarkDef(size=80), strokeWidth=3) + bump_labels
).add_params(bump_highlight, bump_sex).properties(
    width=720, height=400,
    title=alt.Title('1.2 — Bump chart of ranks',
                    subtitle='Top 10 names per decade. Up = rising rank. Click a line to highlight.'),
)


alt.LayerChart(...)

# Problem 2 — Regional effects

## 2.1 — Department choropleth map

In [5]:
def make_choropleth(name, decade):
    filtered = map_data[(map_data['preusuel'] == name) & (map_data['decade'] == decade)][['dpt', 'rate_per_10k', 'nombre']]
    return alt.Chart(geo_inline).mark_geoshape(stroke='white', strokeWidth=0.5).transform_lookup(
        lookup='properties.code',
        from_=alt.LookupData(alt.InlineData(values=filtered.to_dict(orient='records')), 'dpt', ['rate_per_10k', 'nombre']),
    ).encode(
        color=alt.Color('rate_per_10k:Q', scale=alt.Scale(scheme='blues', domainMin=0), title='Births / 10k'),
        tooltip=[alt.Tooltip('properties.nom:N', title='Department'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')],
    ).project('mercator').properties(
        width=620, height=520,
        title=alt.Title(f'2.1 — Choropleth: {name} — {decade}s',
                        subtitle='Darker blue = higher share of births in that department.'),
    )

map_name = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
map_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
map_out = widgets.Output()

def on_map_change(_):
    with map_out:
        clear_output(wait=True)
        display(make_choropleth(map_name.value, map_dec.value))

map_name.observe(on_map_change, names='value')
map_dec.observe(on_map_change, names='value')
display(widgets.HBox([map_name, map_dec]), map_out)
on_map_change(None)


Output()

## 2.9 — Regional flower

In [ ]:
# Schematic layout: central hub + 8 separate petals (Nord, Ouest, IDF, Est, S-O, S-E, Corse, Centre)
REGION_TO_FLOWER = {
    'Hauts-de-France': 'Nord', 'Normandie': 'Nord',
    'Bretagne': 'Ouest', 'Pays-de-la-Loire': 'Ouest',
    'Île-de-France': 'IDF', 'Grand-Est': 'Est',
    'Nouvelle-Aquit.': 'S-O', 'Occitanie': 'S-O',
    'PACA': 'S-E', 'Auvergne-RA': 'Centre',
}
CORSE_DPT = {'2A', '2B', '20'}
FLOWER_ORDER = ['Nord', 'Ouest', 'IDF', 'Est', 'S-O', 'S-E', 'Corse', 'Centre']

def _flower_zone(row):
    if row['dpt'] in CORSE_DPT:
        return 'Corse'
    reg = row.get('region')
    if pd.isna(reg):
        reg = dpt_to_reg.get(row['dpt'])
    return REGION_TO_FLOWER.get(reg)

def _flower_data(name, decade):
    sub = df[(df['preusuel'] == name) & (df['decade'] == decade)].groupby('dpt')['nombre'].sum().reset_index()
    tot = df[df['decade'] == decade].groupby('dpt')['nombre'].sum().reset_index(name='total_births')
    sub = sub.merge(tot, on='dpt', how='left')
    sub['rate_per_10k'] = (sub['nombre'] / sub['total_births'] * 10000).fillna(0)
    sub['region'] = sub['dpt'].map(dpt_to_reg)
    sub['flower'] = sub.apply(_flower_zone, axis=1)
    sub = sub.dropna(subset=['flower'])
    rows = []
    for flower, g in sub.groupby('flower'):
        rate = np.average(g['rate_per_10k'], weights=g['nombre'].clip(lower=1))
        rows.append({'flower': flower, 'rate_per_10k': rate})
    if not rows:
        return pd.DataFrame({'flower': FLOWER_ORDER, 'rate_per_10k': [0.0] * len(FLOWER_ORDER)})
    return pd.DataFrame(rows).set_index('flower').reindex(FLOWER_ORDER).fillna(0).reset_index()

def make_regional_flower(name, decade):
    d = _flower_data(name, decade)
    n = len(FLOWER_ORDER)
    angles = [-np.pi / 2 + i * 2 * np.pi / n for i in range(n)]
    d['angle'] = angles
    r0 = 0.42
    scale = 2.8 / max(d['rate_per_10k'].max(), 0.1)
    d['x1'] = r0 * np.cos(d['angle'])
    d['y1'] = r0 * np.sin(d['angle'])
    r_end = r0 + d['rate_per_10k'] * scale
    d['x2'] = r_end * np.cos(d['angle'])
    d['y2'] = r_end * np.sin(d['angle'])
    d['lx'] = (r_end + 0.22) * np.cos(d['angle'])
    d['ly'] = (r_end + 0.22) * np.sin(d['angle'])
    lim = r0 + d['rate_per_10k'].max() * scale + 0.55
    sc = alt.Scale(domain=[-lim, lim])
    ax = alt.Axis(labels=False, ticks=False, grid=False, domain=False)

    petals = alt.Chart(d).mark_rule(strokeWidth=12, color='#7cb342', opacity=0.92).encode(
        x=alt.X('x1:Q', scale=sc, axis=ax), y=alt.Y('y1:Q', scale=sc, axis=ax),
        x2='x2:Q', y2='y2:Q',
    )
    tips = alt.Chart(d).mark_circle(size=90, color='#2e7d32', stroke='black', strokeWidth=1.2).encode(
        x=alt.X('x2:Q', scale=sc, axis=ax), y=alt.Y('y2:Q', scale=sc, axis=ax),
        tooltip=['flower:N', alt.Tooltip('rate_per_10k:Q', title='Births / 10k', format='.1f')],
    )
    labels = alt.Chart(d).mark_text(fontSize=11, fontWeight='bold').encode(
        x=alt.X('lx:Q', scale=sc, axis=ax), y=alt.Y('ly:Q', scale=sc, axis=ax), text='flower:N',
    )
    hub = alt.Chart(pd.DataFrame({'x': [0], 'y': [0]})).mark_circle(
        size=1400, color='white', stroke='black', strokeWidth=2,
    ).encode(x=alt.X('x:Q', scale=sc, axis=ax), y=alt.Y('y:Q', scale=sc, axis=ax))
    hub_text = alt.Chart(pd.DataFrame({'x': [0], 'y': [0], 't': [f'{name} {decade}']})).mark_text(
        fontSize=11, fontWeight='bold',
    ).encode(x=alt.X('x:Q', scale=sc, axis=ax), y=alt.Y('y:Q', scale=sc, axis=ax), text='t:N')

    return (petals + tips + labels + hub + hub_text).properties(
        width=520, height=520,
        title=alt.Title('2.9 — Fleur régionale',
                        subtitle='Each petal = one macro-region; length = births per 10k.'),
    ).configure_view(stroke=None)

fl_name = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
fl_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
fl_out = widgets.Output()

def on_flower_change(_):
    with fl_out:
        clear_output(wait=True)
        display(make_regional_flower(fl_name.value, fl_dec.value))

fl_name.observe(on_flower_change, names='value')
fl_dec.observe(on_flower_change, names='value')
display(widgets.HBox([fl_name, fl_dec]), fl_out)
on_flower_change(None)


Output()

## 2.4 — Department ranking

In [ ]:
def make_ranking(name, decade, n=20):
    filtered = map_data[(map_data['preusuel'] == name) & (map_data['decade'] == decade)].dropna(subset=['rate_per_10k']).copy()
    filtered['dept_name'] = filtered['dpt'].map(dpt_names_full).fillna('Dept ' + filtered['dpt'])
    filtered = filtered.sort_values('rate_per_10k', ascending=False).head(n)
    return alt.Chart(filtered).mark_bar(color='#2196F3').encode(
        x=alt.X('rate_per_10k:Q', title='Births per 10 000'),
        y=alt.Y('dept_name:N', sort='-x', title='Department'),
        tooltip=[alt.Tooltip('dept_name:N', title='Department'),
                 alt.Tooltip('rate_per_10k:Q', title='Rate / 10k', format='.1f'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')],
    ).properties(width=600, height=480,
                 title=alt.Title(f'2.4 — Top {n} departments for {name} — {decade}s',
                                 subtitle='Readable ranking to complement the choropleth map.'))

rank_name = widgets.Dropdown(options=sorted(top20_map), value='MARIE', description='Name:')
rank_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
rank_out = widgets.Output()

def on_rank_change(_):
    with rank_out:
        clear_output(wait=True)
        display(make_ranking(rank_name.value, rank_dec.value))

rank_name.observe(on_rank_change, names='value')
rank_dec.observe(on_rank_change, names='value')
display(widgets.HBox([rank_name, rank_dec]), rank_out)
on_rank_change(None)


Output()

# Problem 3 — Gender effects

## 3.3 — Baby ratio sketch

In [8]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import PathPatch, Circle, Arc, Patch
from matplotlib.path import Path

# Schematic: three unisex names side by side (Claude, Camille, Sasha)
SKETCH_BABY_NAMES = ['CLAUDE', 'CAMILLE', 'SASHA']
BOY_COLOR, GIRL_COLOR = '#5b9bd5', '#f4b6c8'
BOY_TEXT, GIRL_TEXT = '#2e6eb3', '#c44b7a'
BABY_YMIN, BABY_YMAX = 0.16, 0.94

def _baby_silhouette_path():
    """Sitting-baby outline matching the hand-drawn schematic."""
    verts = [
        (0.50, 0.94), (0.62, 0.92), (0.70, 0.84), (0.72, 0.74), (0.66, 0.66),
        (0.78, 0.60), (0.82, 0.48), (0.76, 0.38), (0.66, 0.32), (0.58, 0.22),
        (0.50, 0.16), (0.42, 0.22), (0.34, 0.32), (0.24, 0.38), (0.18, 0.48),
        (0.22, 0.60), (0.34, 0.66), (0.28, 0.74), (0.30, 0.84), (0.38, 0.92),
        (0.50, 0.94),
    ]
    codes = [Path.MOVETO] + [Path.LINETO] * (len(verts) - 2) + [Path.CLOSEPOLY]
    return Path(verts, codes)

def _paint_baby_fill(ax, path, male_pct, female_pct):
    """Raster mask fill — reliable blue/pink split inside the silhouette."""
    split = BABY_YMIN + (BABY_YMAX - BABY_YMIN) * (female_pct / 100.0)
    n = 600
    xs = np.linspace(0, 1, n)
    ys = np.linspace(0, 1, n)
    X, Y = np.meshgrid(xs, ys)
    inside = path.contains_points(np.column_stack([X.ravel(), Y.ravel()])).reshape(n, n)
    boy = np.array(mcolors.to_rgb(BOY_COLOR))
    girl = np.array(mcolors.to_rgb(GIRL_COLOR))
    rgba = np.zeros((n, n, 4))
    is_boy = inside & (Y >= split)
    is_girl = inside & (Y < split)
    rgba[is_boy, :3] = boy
    rgba[is_boy, 3] = 1
    rgba[is_girl, :3] = girl
    rgba[is_girl, 3] = 1
    ax.imshow(rgba, extent=[0, 1, 0, 1], origin='lower', aspect='equal',
              zorder=1, interpolation='nearest')

def _draw_one_baby(ax, name, male_pct, female_pct):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.text(0.5, 1.05, name.title(), ha='center', va='bottom', fontsize=16,
            fontweight='bold', transform=ax.transAxes)

    path = _baby_silhouette_path()
    _paint_baby_fill(ax, path, male_pct, female_pct)

    ax.add_patch(PathPatch(path, facecolor='none', edgecolor='black', linewidth=2.6, zorder=2))
    ax.add_patch(Circle((0.43, 0.79), 0.022, color='black', zorder=3))
    ax.add_patch(Circle((0.57, 0.79), 0.022, color='black', zorder=3))
    ax.add_patch(Arc((0.50, 0.73), 0.12, 0.07, angle=0, theta1=200, theta2=340,
                     linewidth=2.2, edgecolor='black', fill=False, zorder=3))
    ax.add_patch(Arc((0.63, 0.91), 0.07, 0.05, angle=25, theta1=210, theta2=330,
                     linewidth=2.2, edgecolor='black', fill=False, zorder=3))

    ax.text(0.34, -0.04, f'{male_pct:.0f}%', ha='center', color=BOY_TEXT, fontsize=14,
            fontweight='bold', transform=ax.transAxes)
    ax.text(0.66, -0.04, f'{female_pct:.0f}%', ha='center', color=GIRL_TEXT, fontsize=14,
            fontweight='bold', transform=ax.transAxes)

def _ratio_row(name, decade):
    d = g_piv_dec[(g_piv_dec['preusuel'] == name) & (g_piv_dec['decade'] == decade)]
    if d.empty:
        return 50.0, 50.0
    row = d.iloc[0]
    male_pct = (1 - row['female_ratio']) * 100
    female_pct = row['female_ratio'] * 100
    return male_pct, female_pct

def make_baby_ratio_sketch(decade):
    fig, axes = plt.subplots(1, 3, figsize=(11, 4.2))
    fig.subplots_adjust(top=0.82, bottom=0.12, wspace=0.08)
    for ax, name in zip(axes, SKETCH_BABY_NAMES):
        male_pct, female_pct = _ratio_row(name, decade)
        _draw_one_baby(ax, name, male_pct, female_pct)
    fig.suptitle(
        'Schéma bébé pour ratio filles/garçons prénoms unisexe',
        fontsize=15, fontweight='bold', y=0.98,
    )
    fig.legend(
        handles=[Patch(facecolor=BOY_COLOR, label='Garçons'),
                 Patch(facecolor=GIRL_COLOR, label='Filles')],
        loc='upper right', bbox_to_anchor=(0.98, 0.96), frameon=True, fontsize=11,
    )
    return fig

baby_dec = widgets.Dropdown(options=decades_map, value=2010, description='Decade:')
baby_out = widgets.Output()

def on_baby_change(_):
    with baby_out:
        clear_output(wait=True)
        fig = make_baby_ratio_sketch(baby_dec.value)
        plt.show()

baby_dec.observe(on_baby_change, names='value')
display(baby_dec, baby_out)
on_baby_change(None)


Dropdown(description='Decade:', index=11, options=(1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990,…

Output()

## 3.8 — Gender trajectory

In [9]:
def make_gender_trajectory(name):
    d = g_piv_dec[g_piv_dec['preusuel'] == name].sort_values('decade').copy()
    line = alt.Chart(d).mark_line(point=True, strokeWidth=2.5, color='#6a3d9a').encode(
        x=alt.X('female_ratio:Q', scale=alt.Scale(domain=[0, 1]),
                title='← 100% girls · 50/50 · 100% boys →'),
        y=alt.Y('decade:Q', title='Year', scale=alt.Scale(reverse=True),
                axis=alt.Axis(format='d')),
        tooltip=['preusuel:N', alt.Tooltip('decade:Q', title='Decade'),
                 alt.Tooltip('female_ratio:Q', title='Female ratio', format='.2f'),
                 alt.Tooltip('total:Q', title='Births', format=',')],
    )
    mid = alt.Chart(pd.DataFrame({'x': [0.5]})).mark_rule(strokeDash=[4, 4], color='gray').encode(x='x:Q')
    return (line + mid).properties(
        width=520, height=480,
        title=alt.Title(f'3.8 — Gender trajectory: {name}',
                        subtitle='Vertical axis = time; horizontal drift = shift toward girls or boys.'),
    )

traj_name = widgets.Dropdown(options=sorted(unisex_names) if unisex_names else ['CAMILLE', 'DOMINIQUE'],
                             value=unisex_names[0] if unisex_names else 'CAMILLE', description='Name:')
traj_out = widgets.Output()

def on_traj_change(_):
    with traj_out:
        clear_output(wait=True)
        display(make_gender_trajectory(traj_name.value))

traj_name.observe(on_traj_change, names='value')
display(traj_name, traj_out)
on_traj_change(None)


Dropdown(description='Name:', index=8, options=('CAMILLE', 'CHARLIE', 'DANY', 'DOMINIQUE', 'EDEN', 'LOUISON', …

Output()

## 3.6 — Popularity × mixedness scatterplot

In [10]:
g_embed = g_piv_dec[g_piv_dec['total'] >= 500].copy()
g_embed['decade'] = g_embed['decade'].astype(int)

decade_slider = alt.param(
    name='decade_scatter', value=2010,
    bind=alt.binding_range(min=int(g_embed['decade'].min()), max=int(g_embed['decade'].max()),
                           step=10, name='Decade: '),
)
name_sel = alt.selection_point(fields=['preusuel'], on='click', toggle=True, empty=True)

scatter = alt.Chart(g_embed).mark_circle(opacity=0.7).transform_filter(
    'datum.decade === decade_scatter'
).encode(
    x=alt.X('total:Q', scale=alt.Scale(type='log'), title='Total births (log scale)'),
    y=alt.Y('female_ratio:Q', scale=alt.Scale(domain=[0, 1]),
            title='Female ratio  (0 = all male · 1 = all female)'),
    size=alt.Size('total:Q', scale=alt.Scale(range=[30, 600]), legend=None),
    color=alt.Color('gender_type:N',
        scale=alt.Scale(domain=['Mostly male', 'Mixed (both)', 'Mostly female'],
                        range=['#4878cf', '#6acc65', '#d65f5f']), title='Gender type'),
    opacity=alt.condition(name_sel, alt.value(1.0), alt.value(0.45)),
    tooltip=['preusuel:N', alt.Tooltip('decade:Q', title='Decade'),
             alt.Tooltip('total:Q', title='Total births', format=','),
             alt.Tooltip('female_ratio:Q', title='Female ratio', format='.2f')],
).add_params(decade_slider, name_sel).properties(
    width=720, height=460,
    title=alt.Title('3.6 — Popularity × mixedness scatterplot',
                    subtitle='Each dot = one name. Drag the decade slider; click a dot to highlight.'),
)

hline = alt.Chart(pd.DataFrame({'y': [0.5]})).mark_rule(strokeDash=[5, 3], color='gray', opacity=0.6).encode(y='y:Q')
scatter + hline


alt.LayerChart(...)